# Pipeline1 student results
Select Python (SYMILE-DrFuse). This notebook reads Stage 3 checkpoints and saved reports. Set RUN_DIR to the completed classification run.

In [ ]:
from pathlib import Path
import pandas as pd
import torch
from IPython.display import display
ROOT = next(candidate for parent in (Path.cwd(), *Path.cwd().parents)
            for candidate in (parent, parent / "pipeline1")
            if candidate.name == "pipeline1" and (candidate / "Stage3_CL.py").is_file())
RUN_DIR = ROOT / "runs/classification/version_0"
candidates = list((RUN_DIR / "checkpoints").glob("best*.ckpt"))
CHECKPOINT = max(candidates, key=lambda p:p.stat().st_mtime) if candidates else RUN_DIR / "checkpoints/best.ckpt"
result_file = RUN_DIR / "test_results_student.pt"
if result_file.is_file():
    result = torch.load(result_file, map_location="cpu", weights_only=False)
    CHECKPOINT = Path(result["checkpoint"])
    del result
display(pd.DataFrame([{"item":"Run", "exists":RUN_DIR.exists()}, {"item":"Best checkpoint", "exists":CHECKPOINT.exists()}]))
print("Run", RUN_DIR)
print("Checkpoint", CHECKPOINT)

In [ ]:
saved = torch.load(CHECKPOINT, map_location="cpu", weights_only=False, mmap=True)
assert saved["hyper_parameters"]["stage"] == "classification"
best_epoch = saved["epoch"]
print("Best epoch index", best_epoch)
print("Completed epochs", best_epoch + 1)
print("Optimizer steps", saved["global_step"])
print("Sample limit", saved["hyper_parameters"].get("sample_limit"))
del saved
metrics = pd.read_csv(RUN_DIR / "metrics.csv")
history = metrics.dropna(subset=["val/auroc_macro"]).sort_values(["epoch", "step"]).drop_duplicates("epoch", keep="last")
table = history[["epoch", "val/auroc_macro", "val/ap_macro"]].copy()
table["saved_best_checkpoint"] = table.epoch.eq(best_epoch)
display(table.reset_index(drop=True).round(4))

In [ ]:
for name in ("test_summary.csv", "test_per_finding.csv"):
    path = RUN_DIR / name
    if path.is_file():
        print(name)
        display(pd.read_csv(path).round(4))
    else:
        print("Awaiting Stage 3 test report", path)
for split in ("train", "val", "test"):
    path = RUN_DIR / f"label_counts_{split}.csv"
    if path.is_file():
        print(split)
        display(pd.read_csv(path))